In [ ]:
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm

In [ ]:
# 그래프 시각화를 위한 폰트 설정
font_list = fm.findSystemFonts(fontpaths=None, fontext='ttf')
font_list[:10]  # 일부만 출력
font_path='C:\\Windows\\Fonts'
mpl.rcParams['font.family'] = 'Malgun Gothic'

####    나. 행정동 생활인구 패턴
#####       1) 행정동 요일별 생활인구
#####       2) 행정동 시간대별 생활인구
#####       3) 행정동 성연령별 생활인구
#####       4) 주중 주말 집중도
#####       5) 주민등록인구와 생활인구 비교



행정동, 연도, 월
1.성연령별
2.시간대별
3 .요일별

행정동별 주중,주말비교, 요일
행정동별 시간대별 비교 (연도별은 의미없음,4개년 월평균 혹은 2023년 4개월 평균)
행정동별 연령대별 비교

In [ ]:
siheung_grid_gender = gpd.read_file('../P3_Result/03.SHP/시흥시_100M격자_성연령별_생활인구(행정동).shp', encoding = 'cp949')

In [ ]:
siheung_grid_time = gpd.read_file('../P3_Result/03.SHP/시흥시_100M격자_시간대별_생활인구(행정동).shp', encoding = 'cp949')

In [ ]:
siheung_grid_day = gpd.read_file('../P3_Result/03.SHP/시흥시_100M격자_요일별_생활인구(행정동).shp', encoding = 'cp949')

In [ ]:
c_grid_gender = siheung_grid_gender.copy()
c_grid_time = siheung_grid_time.copy()
c_grid_day = siheung_grid_day.copy()

In [ ]:
def preprocess_pop_gender_age(df):
    # '날짜컬럼'을 문자열 컬럼으로 변경
    df['etl_ym'] = df['etl_ym'].astype(str)

    # 연도 컬럼 생성 (앞 4자리)
    df['Year'] = df['etl_ym'].str[:4]

    # 월 컬럼 생성 (뒤 2자리)
    df['Month'] = df['etl_ym'].str[4:6]

    df.insert(4, 'Year', df.pop('Year')) #컬럼 순서 조정
    df.insert(5, 'Month', df.pop('Month'))

    return df

In [ ]:
# 데이터프레임을 함수에 전달하여 변환
c_grid_gender = preprocess_pop_gender_age(c_grid_gender)
c_grid_time = preprocess_pop_gender_age(c_grid_time)
c_grid_day = preprocess_pop_gender_age(c_grid_day)

In [ ]:
#2023.04 기준 주민등록인구 5세별 남여 인구 데이터
region_pop_10 = pd.read_csv('../P1_Data/02.Pretreatment/주민등록인구_10세별_202304기준(수작업).csv',encoding='cp949')
region_pop_TOTAL = region_pop_10[region_pop_10['항목'] == '총인구수 (명)']
region_pop_TOTAL.reset_index(drop=True)

### 행정동별 인구 추이 비교 (2020년도 1월 ~4월 & 2023년도 1월 ~ 4월)

In [ ]:
compare_204_234 = c_grid_gender.copy()
col = ['M00_09', 'M10_19', 'M20_29', 'M30_39','M40_49', 'M50_59', 'M60_up', 'F00_09', 'F10_19', 'F20_29', 'F30_39','F40_49', 'F50_59', 'F60_up']

compare_204_234['생활인구'] = compare_204_234[col].sum(axis=1)
compare_204_234_groupby = compare_204_234.groupby(['etl_ym','Year','Month','ADM_NM']).sum().reset_index()
compare_204_234_groupby

In [ ]:
# 원하는 연도와 월 범위를 지정하여 행 추출
desired_years = ['2020', '2023']
desired_months = ['01', '02', '03', '04']
filtered_df = compare_204_234_groupby[(compare_204_234_groupby['Year'].isin(desired_years)) & (compare_204_234_groupby['Month'].isin(desired_months))]
filtered_df = filtered_df[['Year','Month','ADM_NM','생활인구']]

# 2020년과 2023년 데이터 분리
df_2020 = filtered_df[filtered_df['Year'] == '2020']
df_2023 = filtered_df[filtered_df['Year'] == '2023']

In [ ]:
# 월별로 데이터 그룹화하고 월평균을 계산
grouped_2020 = df_2020.groupby(['ADM_NM'])['생활인구'].mean().round(1).reset_index()
grouped_2023 = df_2023.groupby(['ADM_NM'])['생활인구'].mean().round(1).reset_index()
# 2020년과 2023년의 월평균 인구수를 병합
merged = pd.merge(grouped_2020, grouped_2023, on=['ADM_NM'], suffixes=('_2020', '_2023'))
merged.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_2020년도_2023년도(1~4월)_인구비교.csv', encoding = 'cp949',index=False)

In [ ]:
HJD_ax_age_pop = compare_204_234_groupby.copy()

desired_years = ['2021', '2022']
desired_months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
filtered_df = HJD_ax_age_pop[(HJD_ax_age_pop['Year'].isin(desired_years)) & (HJD_ax_age_pop['Month'].isin(desired_months))]
filtered_df['total'] = filtered_df[col].sum(axis=1)
filtered_df = filtered_df[['Year','Month','ADM_NM','total']]
filtered_df

In [ ]:
growth_pop = filtered_df.groupby(['Year','Month','ADM_NM'])['total'].sum().round(1).reset_index()
growth_pop2 = growth_pop.groupby(['Year','ADM_NM']).mean().round(1).reset_index()

# pivot을 사용하여 데이터를 재구성
result_df = growth_pop2.pivot(index='ADM_NM', columns='Year', values='total').reset_index()
result_df.columns = ['행정동명','2021년도_월평균유동인구','2022년도_월평균유동인구']
result_df['증가율(%)'] = ((result_df['2022년도_월평균유동인구'] / result_df['2021년도_월평균유동인구'])*100).astype(int)
result_df.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_월평균유동인구_증가율.csv',index = False,encoding='cp949')
result_df

### 행정동별 성연령별 패턴 비교

In [ ]:
dong_pop = c_grid_gender.copy()
gender_age = ['M00_09', 'M10_19', 'M20_29', 'M30_39','M40_49', 'M50_59', 'M60_up', 'F00_09', 'F10_19', 'F20_29', 'F30_39','F40_49', 'F50_59', 'F60_up']

dong_pop['생활인구'] = dong_pop[gender_age].sum(axis=1)
dong_pop_groupby = dong_pop.groupby(['etl_ym','Year','Month','ADM_NM']).sum().reset_index()
dong_pop_groupby = dong_pop_groupby[dong_pop_groupby['etl_ym'] == '202304.0']

gender_age2 = ['M00_09', 'M10_19', 'M20_29', 'M30_39','M40_49', 'M50_59', 'M60_up', 'F00_09', 'F10_19', 'F20_29', 'F30_39','F40_49', 'F50_59', 'F60_up','생활인구']
dong_pop_groupby[gender_age2] = (dong_pop_groupby[gender_age2].div(30)).round(1)


total = dong_pop_groupby[gender_age2].sum()
new_row_data = {'etl_ym': 202304.0,'ADM_NM': '시흥시', **total}
dong_pop_groupby = dong_pop_groupby.append(new_row_data, ignore_index=True)

dong_pop_groupby.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_성연령별(10세)_202304기준_0922.csv', encoding = 'cp949')
dong_pop_groupby

### 행정동별 성별 패턴비교

In [ ]:
# 남여, 총인구 합계
dong_gender_pop = c_grid_gender.copy()

dong_gender_pop['남자'] = dong_gender_pop[['M00_09','M10_19', 'M20_29', 'M30_39', 'M40_49', 'M50_59', 'M60_up']].sum(axis=1)
dong_gender_pop['여자'] = dong_gender_pop[['F00_09','F10_19', 'F20_29', 'F30_39', 'F40_49', 'F50_59', 'F60_up']].sum(axis=1)
dong_gender_pop['생활인구'] = dong_gender_pop[['M00_09','M10_19', 'M20_29', 'M30_39', 'M40_49', 'M50_59', 'M60_up', 'F00_09','F10_19', 'F20_29', 'F30_39', 'F40_49', 'F50_59', 'F60_up']].sum(axis=1)

dong_gender_pop_groupby = dong_gender_pop.groupby(['etl_ym','ADM_NM'])['남자','여자','생활인구'].sum().reset_index()


dong_gender_pop_groupby = dong_gender_pop_groupby[dong_gender_pop_groupby['etl_ym'] == '202304.0']

column = ['남자', '여자', '생활인구']

dong_gender_pop_groupby[column] = (dong_gender_pop_groupby[column].div(30)).round(1) #주민등록인구과 비교예정임으로 나누기 30
dong_gender_pop_groupby.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_생활인구_성별_202304기준_0922.csv', encoding = 'cp949',index=False)
dong_gender_pop_groupby

In [ ]:
#행정동별 연령별 비교(비율산출)
dong_pop2 = c_grid_gender.copy()
gender_age = ['M00_09', 'M10_19', 'M20_29', 'M30_39','M40_49', 'M50_59', 'M60_up', 'F00_09', 'F10_19', 'F20_29', 'F30_39','F40_49', 'F50_59', 'F60_up']

dong_pop['생활인구'] = dong_pop[gender_age].sum(axis=1)
dong_pop_groupby = dong_pop.groupby(['etl_ym','Year','Month','ADM_NM']).sum().round(1).reset_index()
dong_pop_groupby = dong_pop_groupby[dong_pop_groupby['etl_ym'] == '202304.0']
dong_pop_groupby['M00_19'] = dong_pop_groupby['M00_09'] + dong_pop_groupby['M10_19']
dong_pop_groupby['F00_19'] = dong_pop_groupby['F00_09'] + dong_pop_groupby['F10_19']
pre_age_dong = dong_pop_groupby[['etl_ym', 'Year', 'Month', 'ADM_NM', 'M00_19', 'M20_29',
       'M30_39', 'M40_49', 'M50_59', 'M60_up', 'F00_19', 'F20_29',
       'F30_39', 'F40_49', 'F50_59', 'F60_up', '생활인구',]]


pre_age_dong['T00_19'] = pre_age_dong['M00_19'] + pre_age_dong['F00_19']
pre_age_dong['T20_29'] = pre_age_dong['M20_29'] + pre_age_dong['F20_29']
pre_age_dong['T30_39'] = pre_age_dong['M30_39'] + pre_age_dong['F30_39']
pre_age_dong['T40_49'] = pre_age_dong['M40_49'] + pre_age_dong['F40_49']
pre_age_dong['T50_59'] = pre_age_dong['M50_59'] + pre_age_dong['F50_59']
pre_age_dong['T60_up'] = pre_age_dong['M60_up'] + pre_age_dong['F60_up']
pre_age_dong = pre_age_dong[['etl_ym', 'Year', 'Month', 'ADM_NM','생활인구', 'T00_19', 'T20_29', 'T30_39', 'T40_49','T50_59', 'T60_up']]


pre_age_dong['10대_비율'] = (pre_age_dong['T00_19'] / pre_age_dong['생활인구'] * 100).round(1)
pre_age_dong['20대_비율'] = (pre_age_dong['T20_29'] / pre_age_dong['생활인구'] * 100).round(1)
pre_age_dong['30대_비율'] = (pre_age_dong['T30_39'] / pre_age_dong['생활인구'] * 100).round(1)
pre_age_dong['40대_비율'] = (pre_age_dong['T40_49'] / pre_age_dong['생활인구'] * 100).round(1)
pre_age_dong['50대_비율'] = (pre_age_dong['T50_59'] / pre_age_dong['생활인구'] * 100).round(1)
pre_age_dong['60대이상_비율'] = (pre_age_dong['T60_up'] / pre_age_dong['생활인구'] * 100).round(1)

pre_age_dong.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_생활인구_연령별_202304기준_0922.csv', encoding = 'cp949',index=False)
pre_age_dong

### 행정동별 요일별 패턴비교

In [ ]:
#최근 1년
#pd.set_option('display.max_rows', None) #모든 행 출력
c_grid_day_1 = c_grid_day[(c_grid_day['Year'] == '2022') & (c_grid_day['Month'] >= '05') | (c_grid_day['Year'] == '2023') & (c_grid_day['Month'] <= '04')]
#최근 1개월
c_grid_day_30 = c_grid_day[(c_grid_day['Year'] == '2023') & (c_grid_day['Month'] == '04')]
c_grid_day_30

In [ ]:
#c_day_pattern_mean = c_grid_day_1.groupby('ADM_NM').mean().round(1).reset_index()
#c_day_pattern_mean.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_요일패턴(최근1년평균).csv', encoding = 'cp949')
c_day_pattern_mean2 = c_grid_day_30.groupby('ADM_NM').mean().round(1).reset_index()
c_day_pattern_mean2.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_요일패턴(최근1개월평균).csv', encoding = 'cp949')


 ### 행정동별 주중,주말비교

In [ ]:
day_pop = c_grid_day.copy()

#최근 1년
#pd.set_option('display.max_rows', None) #모든 행 출력
day_pop_12 = day_pop[(day_pop['Year'] == '2022') & (day_pop['Month'] >= '05') | (day_pop['Year'] == '2023') & (day_pop['Month'] <= '04')]

# 주중과 주말의 합계
day_pop_12['주중'] = day_pop_12[['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']].sum(axis=1)
day_pop_12['주말'] = day_pop_12[['Saturday', 'Sunday']].sum(axis=1)
day_pop_12 = day_pop_12[['GRID_100M_', 'ADM_CD', 'ADM_NM', 'etl_ym', 'Year', 'Month', '주중', '주말','geometry']]
day_pop_12
# 집중도
day_pop_12['주중_집중도'] = day_pop_12['주중'] / (day_pop_12['주중'] + day_pop_12['주말'])
day_pop_12['주중_집중도'] = day_pop_12['주중_집중도'].fillna(0)  # NaN 값을 0으로 대체합니다.

day_pop_12['주말_집중도'] = day_pop_12['주말'] / (day_pop_12['주말'] + day_pop_12['주중'])
day_pop_12['주말_집중도'] = day_pop_12['주말_집중도'].fillna(0)  # NaN 값을 0으로 대체합니다.
day_pop_12.insert(6, '주중', day_pop_12.pop('주중'))
day_pop_12.insert(7, '주말', day_pop_12.pop('주말'))
day_pop_12.insert(8, '주중_집중도', day_pop_12.pop('주중_집중도'))
day_pop_12.insert(9, '주말_집중도', day_pop_12.pop('주말_집중도'))
day_pop_12

In [ ]:
# 그룹화된 결과에 주중과 주말 컬럼의 합계 계산
focus_day = day_pop_12.groupby(['ADM_NM'])[['주중_집중도', '주말_집중도', '주중', '주말']].agg({'주중': 'sum', '주말': 'sum', '주중_집중도': 'mean', '주말_집중도': 'mean'}).round(2)
focus_day
# 주중과 주말 컬럼을 12로 나누기
focus_day[['주중', '주말']] = focus_day[['주중', '주말']] / 12
focus_day
focus_day.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_주중주말집중도_평균생활인구_0922.csv', encoding = 'cp949')
focus_day

In [ ]:
day_pop_12.to_file('../P3_Result/03.SHP/시흥시_행정동별_주중주말집중도비교(요일별)/주중주말집중도(요일별).gpkg')
day_pop_12.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_주중주말집중도(요일별).csv', encoding = 'cp949' ,index = False)


 ### 행정동별 시간대별 집중도 비교

In [ ]:
#### 행정동별 시간대별 집중도
c_grid_time_202304 = c_grid_time[c_grid_time['etl_ym'] == '202304.0']
c_grid_time_202304

In [ ]:
c_grid_time_202304['생활인구'] = c_grid_time_202304[['0', '1',
       '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14',
       '15', '16', '17', '18', '19', '20', '21', '22', '23']].sum(axis=1)

c_grid_time_202304

In [ ]:
#주요시간 추출
c_grid_time_GRID4 = c_grid_time_202304[['GRID_100M_','ADM_NM','Year','Month','8','12','18','22','생활인구','geometry']]

#시간대별 집중도 계산
c_grid_time_GRID4['오전8시_집중도'] = c_grid_time_GRID4['8'] / c_grid_time_GRID4['생활인구']
c_grid_time_GRID4['오후12시_집중도'] = c_grid_time_GRID4['12'] / c_grid_time_GRID4['생활인구']
c_grid_time_GRID4['오후6시_집중도'] =  c_grid_time_GRID4['18'] / c_grid_time_GRID4['생활인구']
c_grid_time_GRID4['오후10시_집중도'] = c_grid_time_GRID4['22'] / c_grid_time_GRID4['생활인구']
# NaN 값을 0.0으로 채우기
c_grid_time_GRID4['오전8시_집중도'] = c_grid_time_GRID4['오전8시_집중도'].fillna(0.0)
c_grid_time_GRID4['오후12시_집중도'] = c_grid_time_GRID4['오후12시_집중도'].fillna(0.0)
c_grid_time_GRID4['오후6시_집중도'] = c_grid_time_GRID4['오후6시_집중도'].fillna(0.0)
c_grid_time_GRID4['오후10시_집중도'] = c_grid_time_GRID4['오후10시_집중도'].fillna(0.0)


c_grid_time_GRID4.insert(13, 'geometry', c_grid_time_GRID4.pop('geometry'))
c_grid_time_GRID4

In [ ]:
c_grid_time_GRID4.columns

In [ ]:
focus_time = c_grid_time_GRID4.groupby(['ADM_NM'])[['8', '12', '18', '22', '생활인구','오전8시_집중도', '오후12시_집중도', '오후6시_집중도', '오후10시_집중도']].agg\
    ({'8': 'sum', '12': 'sum','18': 'sum','22': 'sum','생활인구': 'sum', '오전8시_집중도': 'mean', '오후12시_집중도':'mean','오후6시_집중도': 'mean','오후10시_집중도': 'mean'}).round(2)

focus_time.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_특정시간집중도_평균생활인구_0922.csv', encoding = 'cp949')
focus_time

In [ ]:
c_grid_time_GRID.to_file('../P3_Result/03.SHP/시흥시_행정동별_시간대별집중도/집중도(시간대별).gpkg')
c_grid_time_GRID.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_집중도(시간대별).csv', encoding = 'cp949' ,index = False)

In [ ]:
c_grid_time_GRID4.to_file('../P3_Result/03.SHP/시흥시_행정동별_시간대별집중도/집중도_특정시간(시간대별).gpkg')
c_grid_time_GRID4.to_csv('../P3_Result/01.CSV/2.행정동_생활인구패턴/행정동별_집중도_특정시간(시간대별).csv', encoding='cp949', index=False)

In [ ]:
#국토부 주민등록인구 100M 격자 데이터 전처리
sihueng_grid100 = gpd.read_file('../P1_Data/01.External/국토부/국토부_시흥시_100m격자_총인구수/(B100)국토통계_인구정보-총 인구 수(전체)-(격자) 100M_경기도 시흥시_202304/nlsp_021001001.shp', encoding='utf-8')
# 'A' 컬럼의 널값에 0 대입
sihueng_grid100['val'].fillna(0, inplace=True)
sihueng_grid100.to_file('../P3_Result/03.SHP/시흥시_거주인구(국토부).gpkg')
sihueng_grid100